# 03 · Image Generation, Editing & Upscaling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/base-batches-workshop/blob/main/notebooks/03-image-generation.ipynb)

Three endpoints:
- `/image/generate` — text → image
- `/image/edit` — image + prompt → image (inpainting / variation)
- `/image/upscale` — image → bigger image

We'll use Stable Diffusion 3.5, then edit, then upscale — all in one notebook.

In [ ]:
%pip install --quiet openai requests python-dotenv rich

In [ ]:
import os, sys

# Try Colab secrets first
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("VENICE_API_KEY")
    os.environ["VENICE_API_KEY"] = api_key
except Exception:
    api_key = os.environ.get("VENICE_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Paste your Venice API key: ").strip()
    os.environ["VENICE_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI(
    api_key=api_key,
    base_url="https://api.venice.ai/api/v1",
)
print("Connected to Venice ✔")

## 1. Generate

The `/image/generate` endpoint isn't OpenAI-shaped, so we'll call it with `requests` directly. Returns base64 PNG.

In [ ]:
import requests, base64
from IPython.display import Image, display

def generate(prompt: str, model: str = "venice-sd35", **kwargs):
    r = requests.post(
        "https://api.venice.ai/api/v1/image/generate",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={
            "model": model,
            "prompt": prompt,
            "width": 1024,
            "height": 1024,
            **kwargs,
        },
        timeout=120,
    )
    r.raise_for_status()
    data = r.json()
    img_b64 = data["images"][0]
    return base64.b64decode(img_b64)

png = generate(
    "A photorealistic gondola at sunset on the Grand Canal, cinematic lighting, 35mm film"
)

with open("venice.png", "wb") as f:
    f.write(png)

display(Image("venice.png"))

## 2. Browse styles

Venice exposes a metadata endpoint with named style presets you can pass as `style_preset`.

In [ ]:
styles = requests.get(
    "https://api.venice.ai/api/v1/image/styles",
    headers={"Authorization": f"Bearer {api_key}"},
).json()

print(f"{len(styles.get('data', []))} styles available. First 12:")
for s in styles.get("data", [])[:12]:
    print(f"  - {s}")

## 3. Generate with a style preset

In [ ]:
png = generate(
    "A futuristic city built on water",
    style_preset="Cinematic",
)
with open("city-cinematic.png", "wb") as f:
    f.write(png)
display(Image("city-cinematic.png"))

## 4. Edit an existing image

Send the original + a prompt, get a new image with the change applied.

In [ ]:
with open("venice.png", "rb") as f:
    src_b64 = base64.b64encode(f.read()).decode()

r = requests.post(
    "https://api.venice.ai/api/v1/image/edit",
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    json={
        "image": src_b64,
        "prompt": "Add a flock of seagulls flying over the canal",
    },
    timeout=120,
)
r.raise_for_status()

edited = base64.b64decode(r.json()["images"][0])
with open("venice-edited.png", "wb") as f:
    f.write(edited)

display(Image("venice-edited.png"))

## 5. Upscale

The upscale endpoint takes an image and returns it at higher resolution with cleaned-up details.

In [ ]:
with open("venice.png", "rb") as f:
    src_b64 = base64.b64encode(f.read()).decode()

r = requests.post(
    "https://api.venice.ai/api/v1/image/upscale",
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    json={"image": src_b64, "scale": 2},
    timeout=180,
)
r.raise_for_status()

upscaled = base64.b64decode(r.json()["images"][0])
with open("venice-upscaled.png", "wb") as f:
    f.write(upscaled)

print(f"Original size: {len(png) / 1024:.0f} KB")
print(f"Upscaled size: {len(upscaled) / 1024:.0f} KB")
display(Image("venice-upscaled.png"))

## What just happened

You generated, restyled, edited, and upscaled an image — all in one notebook, on one provider, on one bill.

The image endpoint is **uncensored by design** (within Venice's terms). For Base Batches use cases this matters most when generating UI mocks, character art, marketing assets — places where overzealous safety filters waste your time.

**Next:** [04 · Audio](./04-audio-tts-stt.ipynb)